# One-GiB File-Backed Performance Profile

This notebook repeats the file-backed/eager comparisons with a
large synthetic HDF5 source. By default it targets a 1 GiB
materialized dynamic spectrum:

- 128 time bins
- 1,048,576 frequency channels
- float64 in-memory frame representation

The HDF5 file size is reported after writing because HDF5 filters
and datatype details affect the exact on-disk size. The source file
is kept under `generated/`; temporary benchmark outputs are deleted
by default so the notebook does not leave several extra GiB behind.

In [ ]:
%matplotlib inline

from pathlib import Path

from IPython import get_ipython
from IPython.display import display
import matplotlib
_ipython = get_ipython()
if _ipython is not None:
    _ipython.run_line_magic("matplotlib", "inline")
    matplotlib.use("module://matplotlib_inline.backend_inline", force=True)

import matplotlib.pyplot as plt
import numpy as np
from astropy import units as u

import setigen as stg

OUT = Path("generated")
OUT.mkdir(exist_ok=True)

np.set_printoptions(precision=4, suppress=True)
print("matplotlib backend:", matplotlib.get_backend())
print("setigen from:", stg.__file__)

In [ ]:
import gc
import os
import time
import tracemalloc

import pandas as pd
import psutil

PROCESS = psutil.Process()
FAST_LARGE_PROFILE = os.environ.get("SETIGEN_FAST_LARGE_PROFILE") == "1"

if FAST_LARGE_PROFILE:
    PROFILE_TCHANS = 16
    PROFILE_FCHANS = 2**15
    PROFILE_CHUNK_BYTES = 512 * 1024
    PROFILE_CONTEXT_WIDTH = 2048
    PROFILE_GUARD_WIDTH = 64
    PROFILE_BOUND_HALF_WIDTH = 256
    SOURCE_STEM = "one_gib_profile_fast"
else:
    PROFILE_TCHANS = 128
    PROFILE_FCHANS = 2**20
    PROFILE_CHUNK_BYTES = 64 * 1024**2
    PROFILE_CONTEXT_WIDTH = 8192
    PROFILE_GUARD_WIDTH = 128
    PROFILE_BOUND_HALF_WIDTH = 2048
    SOURCE_STEM = "one_gib_profile"

PROFILE_DF = 1 * u.Hz
PROFILE_DT = 1 * u.s
PROFILE_FCH1 = (6e9 + PROFILE_FCHANS - 1) * u.Hz

REBUILD_SOURCE = False
KEEP_LARGE_OUTPUTS = False
RUN_UNBOUNDED_SIGNAL = True

source_path = OUT / f"{SOURCE_STEM}_source.h5"

def rss_mib():
    return PROCESS.memory_info().rss / 1024**2

def materialized_gib(dtype=np.float64):
    bytes_ = PROFILE_TCHANS * PROFILE_FCHANS * np.dtype(dtype).itemsize
    return bytes_ / 1024**3

def remove_if_temporary(path):
    path = Path(path)
    if not KEEP_LARGE_OUTPUTS and path.exists():
        path.unlink()

def profile_case(label, func, records, **metadata):
    gc.collect()
    rss_before = rss_mib()
    tracemalloc.start()
    t0 = time.perf_counter()
    result = func()
    elapsed = time.perf_counter() - t0
    _, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    rss_after = rss_mib()

    record = {
        "case": label,
        "elapsed_s": elapsed,
        "rss_delta_mib": rss_after - rss_before,
        "python_peak_mib": peak / 1024**2,
    }
    record.update(metadata)
    records.append(record)
    return result

def show_table(records):
    table = pd.DataFrame(records)
    display(table.round({
        "elapsed_s": 4,
        "rss_delta_mib": 2,
        "python_peak_mib": 2,
        "source_size_mib": 2,
        "output_size_mib": 2,
    }))
    return table

print("fast validation mode:", FAST_LARGE_PROFILE)
print("profile shape:", (PROFILE_TCHANS, PROFILE_FCHANS))
print("materialized float64 size, GiB:", materialized_gib())
print("chunk budget, MiB:", PROFILE_CHUNK_BYTES / 1024**2)
print("source path:", source_path)

In [ ]:
if REBUILD_SOURCE or not source_path.exists():
    print("creating source; this can take time and several GiB of transient RAM in full mode")
    source = stg.Frame(
        tchans=PROFILE_TCHANS,
        fchans=PROFILE_FCHANS,
        df=PROFILE_DF,
        dt=PROFILE_DT,
        fch1=PROFILE_FCH1,
        ascending=False,
        seed=71,
        source_name="One-GiB performance profile source",
    )
    source.add_noise(10, noise_type="chi2")
    source.save_hdf5(source_path)
    del source
    gc.collect()

source_size_mib = source_path.stat().st_size / 1024**2
print("source HDF5 size, MiB:", source_size_mib)
print("source HDF5 size, GiB:", source_size_mib / 1024)

In [ ]:
read_records = []
f0 = PROFILE_FCHANS // 2 - PROFILE_BOUND_HALF_WIDTH
f1 = PROFILE_FCHANS // 2 + PROFILE_BOUND_HALF_WIDTH

def eager_full_load():
    return stg.Frame(waterfall=source_path)

def file_backed_bounded_read():
    with stg.Frame.open(source_path, mode="r", max_chunk_bytes=PROFILE_CHUNK_BYTES) as backed:
        return backed.read_frame(
            f_index_range=(f0, f1),
            t_index_range=(0, backed.tchans),
        )

def file_backed_full_read():
    with stg.Frame.open(source_path, mode="r", max_chunk_bytes=PROFILE_CHUNK_BYTES) as backed:
        return backed.read_frame()

eager = profile_case(
    "eager full load: Frame(waterfall=...)",
    eager_full_load,
    read_records,
    operation="read",
    fchans_read=PROFILE_FCHANS,
    tchans_read=PROFILE_TCHANS,
    source_size_mib=source_size_mib,
)
bounded = profile_case(
    "file-backed bounded read_frame",
    file_backed_bounded_read,
    read_records,
    operation="read",
    fchans_read=f1 - f0,
    tchans_read=PROFILE_TCHANS,
    source_size_mib=source_size_mib,
)
full_backed = profile_case(
    "file-backed full read_frame",
    file_backed_full_read,
    read_records,
    operation="read",
    fchans_read=PROFILE_FCHANS,
    tchans_read=PROFILE_TCHANS,
    source_size_mib=source_size_mib,
)

print("bounded shape:", bounded.shape)
print("full file-backed read matches eager:", np.allclose(full_backed.data, eager.data))
read_table = show_table(read_records)
del eager, bounded, full_backed
gc.collect()

In [ ]:
write_records = []

def eager_rewrite():
    output = OUT / f"{SOURCE_STEM}_eager_rewrite.h5"
    frame = stg.Frame(waterfall=source_path)
    frame.save_hdf5(output)
    output_size_mib = output.stat().st_size / 1024**2
    del frame
    remove_if_temporary(output)
    return output_size_mib

def file_backed_copy():
    output = OUT / f"{SOURCE_STEM}_file_backed_copy.h5"
    with stg.Frame.open_copy(source_path, output, overwrite=True, max_chunk_bytes=PROFILE_CHUNK_BYTES):
        pass
    output_size_mib = output.stat().st_size / 1024**2
    remove_if_temporary(output)
    return output_size_mib

eager_size = profile_case(
    "eager load + save_hdf5",
    eager_rewrite,
    write_records,
    operation="write/copy",
    materializes_full_frame=True,
    source_size_mib=source_size_mib,
)
write_records[-1]["output_size_mib"] = eager_size

copy_size = profile_case(
    "file-backed open_copy",
    file_backed_copy,
    write_records,
    operation="write/copy",
    materializes_full_frame=False,
    source_size_mib=source_size_mib,
)
write_records[-1]["output_size_mib"] = copy_size

write_table = show_table(write_records)

In [ ]:
noise_records = []
noise_config = stg.NoiseEstimationConfig(
    method="sigma_clip",
    context_width=PROFILE_CONTEXT_WIDTH,
    guard_width=PROFILE_GUARD_WIDTH,
)

def local_noise_kwargs(frame):
    return {
        "path": stg.constant_path(
            f_start=frame.get_frequency(frame.fchans // 2),
            drift_rate=0.25 * frame.unit_drift_rate,
        ),
        "f_profile": stg.gaussian_f_profile(width=8 * frame.df),
        "auto_bounding": True,
        "truncate_below": 1e-3,
        "config": noise_config,
    }

def eager_noise_estimation():
    frame = stg.Frame(waterfall=source_path)
    stats = frame.estimate_noise_stats(**local_noise_kwargs(frame))
    del frame
    return stats

def file_backed_noise_estimation():
    with stg.Frame.open(source_path, mode="r", max_chunk_bytes=PROFILE_CHUNK_BYTES) as backed:
        return backed.estimate_noise_stats(**local_noise_kwargs(backed))

eager_noise = profile_case(
    "eager full load + local noise stats",
    eager_noise_estimation,
    noise_records,
    operation="noise estimation",
    materializes_full_frame=True,
    source_size_mib=source_size_mib,
)
backed_noise = profile_case(
    "file-backed local noise stats",
    file_backed_noise_estimation,
    noise_records,
    operation="noise estimation",
    materializes_full_frame=False,
    source_size_mib=source_size_mib,
)

print("eager noise stats:", eager_noise)
print("file-backed noise stats:", backed_noise)
noise_table = show_table(noise_records)

In [ ]:
try:
    output = OUT / f"{SOURCE_STEM}_noise_mutation_attempt.h5"
    with stg.Frame.open_copy(
        source_path,
        output,
        overwrite=True,
        max_chunk_bytes=PROFILE_CHUNK_BYTES,
    ) as backed:
        backed.add_noise(10, noise_type="chi2")
except NotImplementedError as exc:
    print("file-backed add_noise is intentionally unsupported:")
    print(exc)
finally:
    remove_if_temporary(output)

In [ ]:
signal_records = []
FIXED_SIGNAL_LEVEL = 25
TARGET_SIGNAL_SNR = 25

def signal_base_kwargs(frame, *, auto_bounding):
    return {
        "path": stg.constant_path(
            f_start=frame.get_frequency(frame.fchans // 2),
            drift_rate=0.25 * frame.unit_drift_rate,
        ),
        "f_profile": stg.gaussian_f_profile(width=8 * frame.df),
        "bp_profile": stg.constant_bp_profile(level=1),
        "auto_bounding": auto_bounding,
        "truncate_below": 1e-3,
    }

def signal_noise_kwargs(signal_kwargs):
    return {
        "path": signal_kwargs["path"],
        "f_profile": signal_kwargs["f_profile"],
        "auto_bounding": True,
        "truncate_below": signal_kwargs["truncate_below"],
        "config": noise_config,
    }

def injection_kwargs(frame, *, auto_bounding, level_mode):
    kwargs = signal_base_kwargs(frame, auto_bounding=auto_bounding)
    stats = None
    target_snr = np.nan

    if level_mode == "fixed-level":
        level = FIXED_SIGNAL_LEVEL
    elif level_mode == "target-snr":
        stats = frame.estimate_noise_stats(**signal_noise_kwargs(kwargs))
        target_snr = TARGET_SIGNAL_SNR
        level = frame.get_intensity(snr=TARGET_SIGNAL_SNR, noise_stats=stats)
    else:
        raise ValueError(f"Unknown level mode: {level_mode}")

    context_fchans = 0
    excluded_fchans = 0
    if stats is not None:
        context_fchans = stats.context_bounds[1] - stats.context_bounds[0]
        excluded_fchans = stats.excluded_bounds[1] - stats.excluded_bounds[0]

    kwargs["t_profile"] = stg.constant_t_profile(level=level)
    metadata = {
        "level_mode": level_mode,
        "uses_local_noise_stats": stats is not None,
        "target_snr": target_snr,
        "resolved_level": level,
        "noise_mean": np.nan if stats is None else stats.mean,
        "noise_std": np.nan if stats is None else stats.std,
        "noise_samples": 0 if stats is None else stats.n_samples,
        "noise_auto_bounding": False if stats is None else True,
        "noise_context_fchans": context_fchans,
        "noise_excluded_fchans": excluded_fchans,
        "noise_context_bounds": None if stats is None else stats.context_bounds,
        "noise_excluded_bounds": None if stats is None else stats.excluded_bounds,
    }
    return kwargs, metadata

def affected_fchans_from_signal(signal):
    nonzero = np.flatnonzero(np.any(signal != 0, axis=0))
    return int(nonzero.size)

def eager_injection(auto_bounding, level_mode):
    output = OUT / f"{SOURCE_STEM}_eager_{level_mode}_injected_auto_{auto_bounding}.h5"
    frame = stg.Frame(waterfall=source_path)
    kwargs, metadata = injection_kwargs(
        frame,
        auto_bounding=auto_bounding,
        level_mode=level_mode,
    )
    signal = frame.add_signal(**kwargs)
    affected = affected_fchans_from_signal(signal)
    signal_shape = signal.shape
    frame.save_hdf5(output)
    output_size_mib = output.stat().st_size / 1024**2
    del frame, signal
    remove_if_temporary(output)
    return {
        "output_size_mib": output_size_mib,
        "affected_fchans": affected,
        "returned_signal_shape": signal_shape,
        "metadata": metadata,
    }

def file_backed_injection(auto_bounding, level_mode):
    output = OUT / f"{SOURCE_STEM}_file_backed_{level_mode}_injected_auto_{auto_bounding}.h5"
    with stg.Frame.open_copy(
        source_path,
        output,
        overwrite=True,
        max_chunk_bytes=PROFILE_CHUNK_BYTES,
    ) as backed:
        kwargs, metadata = injection_kwargs(
            backed,
            auto_bounding=auto_bounding,
            level_mode=level_mode,
        )
        result = backed.add_signal(
            **kwargs,
            max_chunk_bytes=PROFILE_CHUNK_BYTES,
        )
    output_size_mib = output.stat().st_size / 1024**2
    remove_if_temporary(output)
    return {
        "output_size_mib": output_size_mib,
        "result": result,
        "metadata": metadata,
    }

auto_bounding_values = [True]
if RUN_UNBOUNDED_SIGNAL:
    auto_bounding_values.insert(0, False)

for level_mode in ("fixed-level", "target-snr"):
    label_prefix = "" if level_mode == "fixed-level" else "target-SNR "
    for auto_bounding in auto_bounding_values:
        eager_result = profile_case(
            f"eager {label_prefix}injection auto_bounding={auto_bounding}",
            lambda auto_bounding=auto_bounding, level_mode=level_mode: eager_injection(
                auto_bounding,
                level_mode,
            ),
            signal_records,
            operation="signal injection",
            storage="eager",
            auto_bounding=auto_bounding,
            level_mode=level_mode,
            source_size_mib=source_size_mib,
        )
        signal_records[-1]["affected_fchans"] = eager_result["affected_fchans"]
        signal_records[-1]["returned_signal_shape"] = eager_result["returned_signal_shape"]
        signal_records[-1]["output_size_mib"] = eager_result["output_size_mib"]
        signal_records[-1].update(eager_result["metadata"])

        backed_result = profile_case(
            f"file-backed {label_prefix}injection auto_bounding={auto_bounding}",
            lambda auto_bounding=auto_bounding, level_mode=level_mode: file_backed_injection(
                auto_bounding,
                level_mode,
            ),
            signal_records,
            operation="signal injection",
            storage="file-backed",
            auto_bounding=auto_bounding,
            level_mode=level_mode,
            source_size_mib=source_size_mib,
        )
        fb_result = backed_result["result"]
        signal_records[-1]["affected_fchans"] = fb_result.frequency_slice.stop - fb_result.frequency_slice.start
        signal_records[-1]["time_chunks"] = fb_result.time_chunks
        signal_records[-1]["max_chunk_shape"] = fb_result.max_chunk_shape
        signal_records[-1]["output_size_mib"] = backed_result["output_size_mib"]
        signal_records[-1].update(backed_result["metadata"])

signal_table = show_table(signal_records)

In [ ]:
all_tables = {
    "read": read_table,
    "write": write_table,
    "noise": noise_table,
    "signal": signal_table,
}

for name, table in all_tables.items():
    path = OUT / f"{SOURCE_STEM}_{name}_table.csv"
    table.to_csv(path, index=False)
    print("wrote", path)

def one(table, case):
    match = table.loc[table["case"] == case]
    if len(match) != 1:
        raise ValueError(f"Expected one row for {case!r}, found {len(match)}")
    return match.iloc[0]

def speedup(reference, candidate):
    return float(reference["elapsed_s"] / candidate["elapsed_s"])

def ratio(reference, candidate, column):
    ref = float(reference[column])
    cand = float(candidate[column])
    if cand == 0:
        return np.inf
    return ref / cand

read_eager = one(read_table, "eager full load: Frame(waterfall=...)")
read_bounded = one(read_table, "file-backed bounded read_frame")
read_full_backed = one(read_table, "file-backed full read_frame")
write_eager = one(write_table, "eager load + save_hdf5")
write_copy = one(write_table, "file-backed open_copy")
noise_eager = one(noise_table, "eager full load + local noise stats")
noise_backed = one(noise_table, "file-backed local noise stats")
sig_eager_bounded = one(signal_table, "eager injection auto_bounding=True")
sig_backed_bounded = one(signal_table, "file-backed injection auto_bounding=True")
sig_eager_target_bounded = one(signal_table, "eager target-SNR injection auto_bounding=True")
sig_backed_target_bounded = one(signal_table, "file-backed target-SNR injection auto_bounding=True")

bounded_read_fraction = read_bounded["fchans_read"] / PROFILE_FCHANS
bounded_signal_fraction = sig_backed_bounded["affected_fchans"] / PROFILE_FCHANS
bounded_target_signal_fraction = sig_backed_target_bounded["affected_fchans"] / PROFILE_FCHANS

def bounds_fraction(bounds):
    if bounds is None:
        return np.nan
    return (bounds[1] - bounds[0]) / PROFILE_FCHANS

analysis_rows = [
    {
        "comparison": "bounded file-backed read vs eager full load",
        "wall_time_speedup": speedup(read_eager, read_bounded),
        "python_peak_ratio": ratio(read_eager, read_bounded, "python_peak_mib"),
        "rss_delta_ratio": ratio(read_eager, read_bounded, "rss_delta_mib"),
        "channels_touched_fraction": bounded_read_fraction,
    },
    {
        "comparison": "full file-backed read vs eager full load",
        "wall_time_speedup": speedup(read_eager, read_full_backed),
        "python_peak_ratio": ratio(read_eager, read_full_backed, "python_peak_mib"),
        "rss_delta_ratio": ratio(read_eager, read_full_backed, "rss_delta_mib"),
        "channels_touched_fraction": 1.0,
    },
    {
        "comparison": "file-backed open_copy vs eager load + save",
        "wall_time_speedup": speedup(write_eager, write_copy),
        "python_peak_ratio": ratio(write_eager, write_copy, "python_peak_mib"),
        "rss_delta_ratio": ratio(write_eager, write_copy, "rss_delta_mib"),
        "channels_touched_fraction": 1.0,
    },
    {
        "comparison": "file-backed local noise stats vs eager local noise stats",
        "wall_time_speedup": speedup(noise_eager, noise_backed),
        "python_peak_ratio": ratio(noise_eager, noise_backed, "python_peak_mib"),
        "rss_delta_ratio": ratio(noise_eager, noise_backed, "rss_delta_mib"),
        "channels_touched_fraction": (PROFILE_CONTEXT_WIDTH * 2 + 32) / PROFILE_FCHANS,
    },
    {
        "comparison": "file-backed bounded injection vs eager bounded injection",
        "wall_time_speedup": speedup(sig_eager_bounded, sig_backed_bounded),
        "python_peak_ratio": ratio(sig_eager_bounded, sig_backed_bounded, "python_peak_mib"),
        "rss_delta_ratio": ratio(sig_eager_bounded, sig_backed_bounded, "rss_delta_mib"),
        "channels_touched_fraction": bounded_signal_fraction,
    },
    {
        "comparison": "file-backed bounded target-SNR injection vs eager bounded target-SNR injection",
        "wall_time_speedup": speedup(sig_eager_target_bounded, sig_backed_target_bounded),
        "python_peak_ratio": ratio(sig_eager_target_bounded, sig_backed_target_bounded, "python_peak_mib"),
        "rss_delta_ratio": ratio(sig_eager_target_bounded, sig_backed_target_bounded, "rss_delta_mib"),
        "channels_touched_fraction": bounded_target_signal_fraction,
        "noise_context_fraction": bounds_fraction(sig_backed_target_bounded["noise_context_bounds"]),
    },
]

if RUN_UNBOUNDED_SIGNAL:
    sig_eager_unbounded = one(signal_table, "eager injection auto_bounding=False")
    sig_backed_unbounded = one(signal_table, "file-backed injection auto_bounding=False")
    sig_eager_target_unbounded = one(signal_table, "eager target-SNR injection auto_bounding=False")
    sig_backed_target_unbounded = one(signal_table, "file-backed target-SNR injection auto_bounding=False")
    analysis_rows.extend([
        {
            "comparison": "file-backed unbounded injection vs eager unbounded injection",
            "wall_time_speedup": speedup(sig_eager_unbounded, sig_backed_unbounded),
            "python_peak_ratio": ratio(sig_eager_unbounded, sig_backed_unbounded, "python_peak_mib"),
            "rss_delta_ratio": ratio(sig_eager_unbounded, sig_backed_unbounded, "rss_delta_mib"),
            "channels_touched_fraction": 1.0,
        },
        {
            "comparison": "file-backed bounded injection vs file-backed unbounded injection",
            "wall_time_speedup": speedup(sig_backed_unbounded, sig_backed_bounded),
            "python_peak_ratio": ratio(sig_backed_unbounded, sig_backed_bounded, "python_peak_mib"),
            "rss_delta_ratio": ratio(sig_backed_unbounded, sig_backed_bounded, "rss_delta_mib"),
            "channels_touched_fraction": bounded_signal_fraction,
        },
        {
            "comparison": "file-backed unbounded target-SNR injection vs eager unbounded target-SNR injection",
            "wall_time_speedup": speedup(sig_eager_target_unbounded, sig_backed_target_unbounded),
            "python_peak_ratio": ratio(sig_eager_target_unbounded, sig_backed_target_unbounded, "python_peak_mib"),
            "rss_delta_ratio": ratio(sig_eager_target_unbounded, sig_backed_target_unbounded, "rss_delta_mib"),
            "channels_touched_fraction": 1.0,
            "noise_context_fraction": bounds_fraction(sig_backed_target_unbounded["noise_context_bounds"]),
        },
        {
            "comparison": "file-backed bounded target-SNR injection vs file-backed unbounded target-SNR injection",
            "wall_time_speedup": speedup(sig_backed_target_unbounded, sig_backed_target_bounded),
            "python_peak_ratio": ratio(sig_backed_target_unbounded, sig_backed_target_bounded, "python_peak_mib"),
            "rss_delta_ratio": ratio(sig_backed_target_unbounded, sig_backed_target_bounded, "rss_delta_mib"),
            "channels_touched_fraction": bounded_target_signal_fraction,
            "noise_context_fraction": bounds_fraction(sig_backed_target_bounded["noise_context_bounds"]),
        },
    ])

analysis_table = pd.DataFrame(analysis_rows)
display(analysis_table.round({
    "wall_time_speedup": 2,
    "python_peak_ratio": 2,
    "rss_delta_ratio": 2,
    "channels_touched_fraction": 6,
    "noise_context_fraction": 6,
}))

analysis_path = OUT / f"{SOURCE_STEM}_analysis_table.csv"
analysis_table.to_csv(analysis_path, index=False)
print("wrote", analysis_path)

In [ ]:
from IPython.display import Markdown

def fmt_speed(value):
    return f"{value:.1f}x"

def fmt_pct(value):
    return f"{100 * value:.4f}%"

read_speed = speedup(read_eager, read_bounded)
noise_speed = speedup(noise_eager, noise_backed)
copy_speed = speedup(write_eager, write_copy)
bounded_signal_speed = speedup(sig_eager_bounded, sig_backed_bounded)
bounded_target_signal_speed = speedup(sig_eager_target_bounded, sig_backed_target_bounded)
target_context_fraction = bounds_fraction(sig_backed_target_bounded["noise_context_bounds"])

unbounded_sentence = ""
if RUN_UNBOUNDED_SIGNAL:
    file_backed_bound_speed = speedup(sig_backed_unbounded, sig_backed_bounded)
    file_backed_target_bound_speed = speedup(
        sig_backed_target_unbounded,
        sig_backed_target_bounded,
    )
    unbounded_sentence = (
        f" Bounded file-backed injection was {fmt_speed(file_backed_bound_speed)} "
        "faster than file-backed unbounded injection. Bounded target-SNR "
        f"injection was {fmt_speed(file_backed_target_bound_speed)} faster "
        "than file-backed unbounded target-SNR injection."
    )

interpretation = f'''
### One-GiB Interpretation

- The source target is {materialized_gib():.2f} GiB as a materialized
  float64 frame. The actual HDF5 file written here is
  {source_size_mib / 1024:.2f} GiB.

- Bounded file-backed reads touched {int(read_bounded["fchans_read"]):,}
  of {PROFILE_FCHANS:,} channels ({fmt_pct(bounded_read_fraction)}) and
  were {fmt_speed(read_speed)} faster than eager full loading.

- Full file-backed reads are expected to be close to eager full loading
  because both paths must materialize the full observation.

- File-backed `open_copy(...)` was {fmt_speed(copy_speed)} faster than
  eager load plus `save_hdf5(...)` and did not materialize the full frame
  in Python.

- File-backed local noise/SNR context estimation was
  {fmt_speed(noise_speed)} faster than the eager path while producing the
  same statistics. This is the scientifically relevant mode for narrowband
  injections because SNR should be estimated locally around the signal
  track.

- Target-SNR injection rows estimate local sigma-clipped noise statistics
  before mutation, then call `get_intensity(snr=..., noise_stats=...)`.
  The bounded file-backed target-SNR row used
  {int(sig_backed_target_bounded["noise_samples"]):,} noise samples from
  {fmt_pct(target_context_fraction)} of the frequency axis and resolved
  SNR {sig_backed_target_bounded["target_snr"]:.1f} to an intensity level
  of {sig_backed_target_bounded["resolved_level"]:.4g}. These timings
  intentionally include the local noise-estimation cost.

- Bounded file-backed signal injection touched
  {int(sig_backed_bounded["affected_fchans"]):,} of {PROFILE_FCHANS:,}
  channels ({fmt_pct(bounded_signal_fraction)}) and was
  {fmt_speed(bounded_signal_speed)} faster than eager bounded injection.
  {unbounded_sentence}

- Bounded file-backed target-SNR injection touched
  {int(sig_backed_target_bounded["affected_fchans"]):,} of
  {PROFILE_FCHANS:,} channels ({fmt_pct(bounded_target_signal_fraction)})
  and was {fmt_speed(bounded_target_signal_speed)} faster than eager
  bounded target-SNR injection.

- Timings are still local wall-clock measurements. For reporting, prefer
  pairing speedups with the source file size, materialized frame size,
  channel fraction touched, and whether the operation materialized the
  whole frame.
'''

display(Markdown(interpretation))